# Dataset Consolidado

Este notebook consolida los aprendizajes de `01_exploration.ipynb` y `02_clustering.ipynb` en un dataset limpio y dedupado listo para la fase de modelado.

## Aprendizajes que aplicamos

**De `01_exploration.ipynb`:**
- De 586 filas originales, solo 297 tienen DFM analysis completo (usamos `part_volume_cm3` como indicador).
- 5 features son **constantes (siempre 0)** y se descartan: `face_orientation_distribution.X-_pct/Y-_pct/Z-_pct`, `undercut_count`, `inaccessible_feature_count`.
- 2 pares son **redundantes (|corr|>0.9)**: `direction_count` ≡ `min_setups_3axis`, `thin_ratio` ↔ `bbox_ratio_3`.
- `angled_hole_count` es muy skewed → `log1p`.
- `bounding_box` absoluto se transforma en proporciones invariantes a tamaño (`bbox_ratio_2`, `bbox_ratio_3`).
- `face_orientation_distribution.*` se convierten a porcentajes (dividir por suma).

**De `02_clustering.ipynb`:**
- Descartamos `part_type` como columna pero **conservamos todas las piezas con DFM** (rotational + complex_3d + plate + prismatic).
- Varias piezas aparecen cotizadas múltiples veces (mismo `product_name`) → dedupe (preferimos la cotización awarded más reciente).

## Flujo
1. Cargar CSV original
2. Feature engineering (ratios bbox, % face orientations, log angled holes)
3. Filtrar a filas con DFM
4. Seleccionar columnas finales (shape features + categoricals + aux + size), descartando `part_type`
5. Dedupe por `product_name` (keep awarded más reciente)
6. Guardar

In [34]:
import numpy as np
import pandas as pd

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 200)

CSV_PATH = 'manufacturing_quotes_data.csv'

# ─────────────────────────────────────────────────────────────────────
# FLAG: controla si deduplicamos por product_name al final
#   True  → 1 fila por pieza (cotización awarded más reciente)
#           útil para clustering de forma, dataset compacto
#   False → todas las cotizaciones de todos los partners por pieza
#           útil para modelado de precio (más datos, partner_name como feature)
# ─────────────────────────────────────────────────────────────────────
DEDUPE_BY_PRODUCT_NAME = False

## 1. Cargar CSV original

In [35]:
df = pd.read_csv(CSV_PATH, skipinitialspace=True)
df.columns = df.columns.str.strip()
# Limpiamos espacios en strings categóricos (notamos trailing spaces en axis_recommendation, etc.)
for col in df.select_dtypes(include='object').columns:
    df[col] = df[col].str.strip() if df[col].dtype == 'object' else df[col]
print(f'Dataset original: {df.shape}')

Dataset original: (586, 91)


## 1b. Normalizar `partner_country` → {MX, Asia}

Los valores originales incluyen variantes regionales (SJR = San Juan del Río, "México, NL", etc.) y múltiples países asiáticos. Los unificamos en dos buckets para reducir cardinalidad y capturar la señal real (costo-país).

In [36]:
MX_PATTERNS = [
    'sjr', 'san juan', 'mexico', 'méxico', 'mx', 'mex',
    'queretaro', 'querétaro', 'guadalajara', 'monterrey',
    'jalisco', 'puebla', 'tlaxcala',
]
ASIA_PATTERNS = ['china', 'japan', 'japón', 'korea', 'taiwan', 'vietnam', 'india', 'thailand', 'asia']


def normalize_partner_country(value):
    if pd.isna(value):
        return value
    v = str(value).strip().lower()
    if any(p in v for p in MX_PATTERNS):
        return 'MX'
    if any(p in v for p in ASIA_PATTERNS):
        return 'Asia'
    return value  # sin mapear — lo dejamos visible para revisar


print('ANTES:')
print(df['partner_country'].value_counts(dropna=False).to_string())

df['partner_country'] = df['partner_country'].apply(normalize_partner_country)

print('\nDESPUÉS:')
print(df['partner_country'].value_counts(dropna=False).to_string())

# Alerta si quedaron valores sin mapear
unmapped = set(df['partner_country'].dropna().unique()) - {'MX', 'Asia'}
if unmapped:
    print(f'\n⚠️  Valores sin mapear (revisar): {unmapped}')
else:
    print('\n✅ Todos los valores mapeados a {MX, Asia}.')

ANTES:
partner_country
China               340
SJR                  94
MX                   74
Querétaro            16
NaN                  15
Tlaxcala, México     15
Asia                 12
Japan                 8
Edo. Mex              3
México                3
India                 3
México, NL            1
Guadalajara           1
mx                    1

DESPUÉS:
partner_country
Asia    363
MX      208
NaN      15

✅ Todos los valores mapeados a {MX, Asia}.


## 2. Feature engineering

Derivamos los features *dimensionless* que usaremos para modelado de forma.

In [37]:
# 2.1 Bounding box → proporciones ordenadas (invariantes a rotación y tamaño)
bbox_cols = ['metadata.bounding_box_mm.x', 'metadata.bounding_box_mm.y', 'metadata.bounding_box_mm.z']
bbox = df[bbox_cols].values
with np.errstate(invalid='ignore'):
    bbox_max = np.nanmax(bbox, axis=1, keepdims=True)
    bbox_max_safe = np.where((bbox_max == 0) | np.isnan(bbox_max), 1, bbox_max)
    bbox_ratios = bbox / bbox_max_safe
    bbox_ratios_sorted = -np.sort(-bbox_ratios, axis=1)
df['bbox_ratio_2'] = bbox_ratios_sorted[:, 1]
df['bbox_ratio_3'] = bbox_ratios_sorted[:, 2]

# 2.2 Face orientations → porcentajes (solo +X, +Y, +Z; las negativas siempre son 0)
face_pos = ['face_orientation_distribution.X+', 'face_orientation_distribution.Y+', 'face_orientation_distribution.Z+']
face_totals = df[face_pos].sum(axis=1)
face_totals_safe = face_totals.replace(0, 1)
for c in face_pos:
    df[c + '_pct'] = df[c] / face_totals_safe

# 2.3 angled_hole_count → log1p
df['angled_hole_count_log'] = np.log1p(df['angled_hole_count'])

print('Features derivados: bbox_ratio_2, bbox_ratio_3, *_pct (3), angled_hole_count_log')

Features derivados: bbox_ratio_2, bbox_ratio_3, *_pct (3), angled_hole_count_log


C:\Users\Imanol\AppData\Local\Temp\ipykernel_1300\3577782804.py:5: RuntimeWarning: All-NaN slice encountered
  bbox_max = np.nanmax(bbox, axis=1, keepdims=True)


## 3. Filtrar: filas con DFM (todas las piezas, sin filtrar por tipo)

In [38]:
has_dfm = df['part_volume_cm3'].notna()

print(f'Original:  {len(df)}')
print(f'Con DFM:   {has_dfm.sum()}')

df_dfm = df[has_dfm].copy().reset_index(drop=True)

print(f'\nDistribución de part_type en filas con DFM:')
print(df_dfm['part_type'].value_counts(dropna=False).to_string())

Original:  586
Con DFM:   297

Distribución de part_type en filas con DFM:
part_type
rotational    252
complex_3d     29
plate          14
prismatic       1
NaN             1


## 4. Columnas finales

Descartamos `part_type` y las 7 columnas inútiles/redundantes identificadas en notebooks anteriores.

In [39]:
# Identificadores + metadata del quote
AUX_COLS = [
    'manufacturing_quote_id', 'manufacturing_quote_created_at',
    'line_item_id', 'radii_id', 'product_name',
    'partner_name', 'partner_country', 'quote_is_awarded',
    'actual_partner_quoted_price', 'actual_partner_quoted_quantity',
]

# Categóricas de especificación (útiles para fase 2)
SPEC_CATEGORICAL = ['material_title', 'technology', 'tolerance_tier', 'finishing', 'heat_treatment']

# Categórica de forma
SHAPE_CATEGORICAL = ['axis_recommendation']

# Shape features dimensionless (12 — feature set limpio)
SHAPE_NUMERIC = [
    'aspect_ratio', 'plane_ratio', 'cyl_ratio', 'complex_ratio',
    'material_removal_pct',
    'bbox_ratio_2', 'bbox_ratio_3',
    'face_orientation_distribution.X+_pct',
    'face_orientation_distribution.Y+_pct',
    'face_orientation_distribution.Z+_pct',
    'direction_count',
    'angled_hole_count_log',
]

# Size features (para fase 2 — no forma)
SIZE_NUMERIC = [
    'part_volume_cm3', 'weight_g', 'stock_volume_cm3', 'removed_volume_cm3',
    'metadata.bounding_box_mm.x', 'metadata.bounding_box_mm.y', 'metadata.bounding_box_mm.z',
    'total_holes', 'faces_qty', 'metadata.max_hole_depth_mm',
]

KEEP = AUX_COLS + SPEC_CATEGORICAL + SHAPE_CATEGORICAL + SHAPE_NUMERIC + SIZE_NUMERIC
KEEP = [c for c in KEEP if c in df_dfm.columns]

df_final = df_dfm[KEEP].copy()

print(f'Columnas seleccionadas: {len(df_final.columns)}')
print(f'  - Aux: {len(AUX_COLS)}')
print(f'  - Spec categoricals: {len(SPEC_CATEGORICAL)}')
print(f'  - Shape categoricals: {len(SHAPE_CATEGORICAL)}')
print(f'  - Shape numeric (modelado): {len(SHAPE_NUMERIC)}')
print(f'  - Size numeric (fase 2): {len(SIZE_NUMERIC)}')
print(f'\nShape final pre-dedupe: {df_final.shape}')
print(f'\n⚠️  part_type NO está en el dataset final (por diseño).')

Columnas seleccionadas: 38
  - Aux: 10
  - Spec categoricals: 5
  - Shape categoricals: 1
  - Shape numeric (modelado): 12
  - Size numeric (fase 2): 10

Shape final pre-dedupe: (297, 38)

⚠️  part_type NO está en el dataset final (por diseño).


## 5. Dedupe por `product_name` (opcional)

Varias piezas tienen múltiples quotes de distintos partners (ej. "Disco", "cilindro"). Según el caso de uso:

- **`DEDUPE_BY_PRODUCT_NAME = True`** → conservamos 1 fila por pieza, priorizando:
  1. Cotizaciones awarded (`quote_is_awarded=True`)
  2. Más reciente (`manufacturing_quote_created_at` más alto)

  Útil para análisis de **forma** (clustering, tipos de piezas) donde no queremos sobre-representar familias cotizadas muchas veces.

- **`DEDUPE_BY_PRODUCT_NAME = False`** → conservamos **todas las cotizaciones**.

  Útil para **modelar precio**, donde cada partner aporta una cotización distinta sobre la misma pieza. Conservamos `partner_name` y `quote_is_awarded` como features útiles.

In [40]:
# Cuántos product_name se repiten
dup_counts = df_final['product_name'].value_counts()
repeated = dup_counts[dup_counts > 1]
print(f'product_name únicos: {dup_counts.shape[0]}')
print(f'product_name repetidos: {len(repeated)}')
print(f'Filas duplicadas (exceso): {repeated.sum() - len(repeated)}')
print(f'\nTop 10 más repetidos:')
print(repeated.head(10).to_string())

product_name únicos: 156
product_name repetidos: 60
Filas duplicadas (exceso): 141

Top 10 más repetidos:
product_name
Disco                           9
soprte1                         6
Soporte Inferior                6
89002300-01-000-04-00 cortes    6
89002300-01-000-02-00 cortes    6
89002300-01-000-12-00 cortes    6
89002300-01-000-22-00 cortes    6
89002300-01-000-01-00 cortes    6
cilindro                        6
placa tapa                      6


In [41]:
if DEDUPE_BY_PRODUCT_NAME:
    # Convertimos created_at a datetime para ordenar
    df_final['_created_dt'] = pd.to_datetime(df_final['manufacturing_quote_created_at'], errors='coerce')

    # Ordenamos: awarded primero, luego más reciente primero
    df_sorted = df_final.sort_values(
        by=['quote_is_awarded', '_created_dt'],
        ascending=[False, False],
        na_position='last',
    )

    df_dedup = df_sorted.drop_duplicates(subset='product_name', keep='first').copy()
    df_dedup = df_dedup.drop(columns='_created_dt').reset_index(drop=True)

    print(f'Pre-dedupe:  {len(df_final)} filas')
    print(f'Post-dedupe: {len(df_dedup)} filas')
    print(f'Eliminadas:  {len(df_final) - len(df_dedup)} duplicados')

    assert df_dedup['product_name'].nunique() == len(df_dedup), 'Aún hay duplicados!'
else:
    df_dedup = df_final.copy()
    print(f'⏭️  Dedupe desactivado (DEDUPE_BY_PRODUCT_NAME=False)')
    print(f'   Conservando las {len(df_dedup)} filas con todas las cotizaciones por pieza.')
    print(f'   Piezas únicas: {df_dedup["product_name"].nunique()}')
    print(f'   Promedio de quotes por pieza: {len(df_dedup) / df_dedup["product_name"].nunique():.1f}')

# Drop columnas de metadata del quote que no aportan al modelado.
# Cuando NO dedupeamos, conservamos partner_name y quote_is_awarded como features.
DROP_COLS = [
    'manufacturing_quote_id', 'manufacturing_quote_created_at',
    'line_item_id',
]
if DEDUPE_BY_PRODUCT_NAME:
    DROP_COLS += ['partner_name', 'quote_is_awarded']

df_dedup = df_dedup.drop(columns=[c for c in DROP_COLS if c in df_dedup.columns])
print(f'\nColumnas dropeadas: {DROP_COLS}')
print(f'Shape final: {df_dedup.shape}')

⏭️  Dedupe desactivado (DEDUPE_BY_PRODUCT_NAME=False)
   Conservando las 297 filas con todas las cotizaciones por pieza.
   Piezas únicas: 156
   Promedio de quotes por pieza: 1.9

Columnas dropeadas: ['manufacturing_quote_id', 'manufacturing_quote_created_at', 'line_item_id']
Shape final: (297, 35)


## 5b. Filtrar precios inválidos y outliers

Antes de guardar aplicamos dos filtros sobre `actual_partner_quoted_price` y `actual_partner_quoted_quantity`:\n\n1. **Precios inválidos**: `price > 1 USD` y `quantity > 0` (descarta quotes con precio $0 o cantidad nula, que son errores de captura y rompen métricas como MAPE).\n2. **Outliers**: regla IQR sobre `log1p(unit_price)` con multiplicador 1.5×. Usamos `unit_price = price / quantity` (elimina efecto de lote) y `log1p` (normaliza el skew). Esto corta valores extremos que distorsionan el modelo.

In [42]:
# 1. Filtrar precios y cantidades inválidas
before = len(df_dedup)
df_dedup = df_dedup[
    (df_dedup['actual_partner_quoted_price'] > 1)
    & (df_dedup['actual_partner_quoted_quantity'] > 0)
].reset_index(drop=True)
print(f'Filtro price>1 & quantity>0: {before} → {len(df_dedup)} filas ({before - len(df_dedup)} eliminadas)')

# 2. Outliers: IQR sobre log1p(unit_price) con 1.5×IQR
unit_price = df_dedup['actual_partner_quoted_price'] / df_dedup['actual_partner_quoted_quantity']
log_up = np.log1p(unit_price)
q1, q3 = log_up.quantile([0.25, 0.75])
iqr = q3 - q1
lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr

mask_ok = (log_up >= lower) & (log_up <= upper)
removed = df_dedup.loc[~mask_ok, ['product_name', 'partner_country', 'actual_partner_quoted_price', 'actual_partner_quoted_quantity']].copy()
removed['unit_price'] = unit_price[~mask_ok].round(2).values

before = len(df_dedup)
df_dedup = df_dedup[mask_ok].reset_index(drop=True)
print(f'Filtro outliers (IQR 1.5× sobre log1p(unit_price)): {before} → {len(df_dedup)} filas ({before - len(df_dedup)} eliminadas)')
print(f'  Rango válido unit_price: [${np.expm1(lower):.2f}, ${np.expm1(upper):.2f}]')

if len(removed) > 0:
    print(f'\nOutliers eliminados:')
    print(removed.to_string(index=False))

print(f'\n✅ Shape final post-limpieza: {df_dedup.shape}')

Filtro price>1 & quantity>0: 297 → 288 filas (9 eliminadas)
Filtro outliers (IQR 1.5× sobre log1p(unit_price)): 288 → 288 filas (0 eliminadas)
  Rango válido unit_price: [$-0.99, $57880.45]

✅ Shape final post-limpieza: (288, 35)


## 6. Guardar

In [43]:
df_dedup.to_parquet('manufacturing_quotes_clean.parquet', index=False)
df_dedup.to_csv('manufacturing_quotes_clean.csv', index=False)

print(f'Guardado: manufacturing_quotes_clean.parquet + .csv')
print(f'Shape final: {df_dedup.shape}')
print(f'\nColumnas ({len(df_dedup.columns)}):')
for c in df_dedup.columns:
    print(f'  {c}')

Guardado: manufacturing_quotes_clean.parquet + .csv
Shape final: (288, 35)

Columnas (35):
  radii_id
  product_name
  partner_name
  partner_country
  quote_is_awarded
  actual_partner_quoted_price
  actual_partner_quoted_quantity
  material_title
  technology
  tolerance_tier
  finishing
  heat_treatment
  axis_recommendation
  aspect_ratio
  plane_ratio
  cyl_ratio
  complex_ratio
  material_removal_pct
  bbox_ratio_2
  bbox_ratio_3
  face_orientation_distribution.X+_pct
  face_orientation_distribution.Y+_pct
  face_orientation_distribution.Z+_pct
  direction_count
  angled_hole_count_log
  part_volume_cm3
  weight_g
  stock_volume_cm3
  removed_volume_cm3
  metadata.bounding_box_mm.x
  metadata.bounding_box_mm.y
  metadata.bounding_box_mm.z
  total_holes
  faces_qty
  metadata.max_hole_depth_mm


In [44]:
# Sanity check final — describe de shape features
df_dedup[SHAPE_NUMERIC].describe().T.round(3)

,count,mean,std,min,25%,50%,75%,max
aspect_ratio,288.0,5.917,5.919,1.00,2.245,4.225,6.895,33.550
plane_ratio,288.0,0.305,0.164,0.00,0.171,0.308,0.423,0.846
cyl_ratio,288.0,0.454,0.167,0.00,0.333,0.468,0.568,0.852
complex_ratio,288.0,0.054,0.124,0.00,0.000,0.000,0.020,0.635
material_removal_pct,288.0,67.758,20.108,0.00,57.000,71.950,83.025,98.800
bbox_ratio_2,279.0,0.640,0.298,0.09,0.367,0.608,0.969,1.000
bbox_ratio_3,279.0,0.308,0.219,0.03,0.144,0.236,0.435,0.882
face_orientation_distribution.X+_pct,279.0,0.295,0.250,0.00,0.069,0.312,0.381,1.000
face_orientation_distribution.Y+_pct,279.0,0.279,0.276,0.00,0.026,0.250,0.333,1.000
face_orientation_distribution.Z+_pct,279.0,0.427,0.294,0.00,0.250,0.353,0.625,1.000
